# IIR Response Generation Pipeline

Generates simulated student responses to three short stories (**Cat**, **Lying**, **Stealing**) — one model call per student — using Google Gemini via Vertex AI. Each student is assigned an
IIR developmental waypoint (WAYPOINT0–WAYPOINT3) and a format (comic or text); results are written to
`IIR_outputs/` as a CSV.

**Before running:**
1. Authenticate once in a terminal: `gcloud auth application-default login`
2. Set your project (PowerShell): `$env:PROJECT_CODE = "your-gcp-project-id"`
3. Run the cells top to bottom.


In [2]:
# ============================================================
# Imports & configuration
# ============================================================
import os
import json
import time
import random
from functools import lru_cache

import pandas as pd
import vertexai
from vertexai.generative_models import GenerativeModel, Part

# Output locations
OUTPUT_DIR = "IIR_outputs"
OUTPUT_CSV = "IIR_raw_answers_Master4.csv"
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Model
MODEL_NAME = "gemini-3.1-pro-preview"

# Authenticate via Application Default Credentials (run once in a terminal):
#   gcloud auth application-default login
# The project ID is read from the PROJECT_CODE environment variable so it is
# never hard-coded. Set it once before running, e.g. in PowerShell:
#   $env:PROJECT_CODE = "your-gcp-project-id"
PROJECT_CODE = os.environ.get("PROJECT_CODE")
vertexai.init(project=PROJECT_CODE, location="global")


In [3]:
# ============================================================
# Student profiles & system prompt
# ============================================================
profiles = {
    "IIR0": {
        "label": "0 — No inference",
        "definition": (
            "Students do not generate an inference nor try to answer the question.  Instead, they say 'I don't know' or make a nonsensical statement (e.g., stating irrelevant world knowledge, unrelated to the story). In other words, they fail to make a coherent link  between the question, the text, or their world knowledge."
        )
    },
    "IIR1": {
        "label": "1 —Text-Implicit",
        "definition": (
            "Students generate an inference connecting the question only with the information explicitly stated in the story. Use of world-knowledge is absent. This signals student inference is limited to text-base representation; without the schema/background knowledge activation, their inference may be thin, lacking elaboration of the “why” and the “how” behind the narrative."

        )
    },
    "IIR2": {
        "label": "2 — Script-Implicit",
        "definition": (
            "Students generate an inference connecting the question primarily with their background knowledge. Use of text-base information is implicit. This signals their 'script-based' reasoning—they answer based on how the world usually operates without explicitly integrating how the story specifically describes it."
        )
    },
    "IIR3": {
        "label": "3 — Combination",
        "definition": (
             "Students generate an inference connecting the question with both world/background knowledge AND information explicitly mentioned in the text (text-based information). In other words, students go beyond merely identifying text-base information,  anchoring it within a broader real-world schema (i.e., background knowledge) to explain the why or how behind the narrative. This achieves the highest level of coherence (“the situation model”, Kintsch, 1988, 1998) because the student integrates the text-base information and their prior knowledge to generate a novel, unified interpretation."

        )
    },
}


System_Prompt = ("You are an experienced high school / middle school instructor with deep expertise in how students develop Inferential Integration in Reading (IIR) — "
"the ability to understand information from a story with world knowledge to construct meaning. "
"Based on your teaching experience, predict the answer that a specific student would give, based on their respective performance waypoint. "
"Do not explain your reasoning. Only output the student's response."
"\n Each student response must follow one of the 4 descriptive waypoints regarding student performance:"
"\n IIR0: No meaningful response provided."
    "\n Students do not generate an inference nor try to answer the question.  Instead, they say that they don't know or make a nonsensical statement (e.g., stating irrelevant world knowledge, unrelated to the story). In other words, they fail to make a coherent link  between the question, the text, or their world knowledge."
"\n IIR1: Text-Base Only (Local)"
    "\n Students generate an inference connecting the question only with the information explicitly stated in the story. Use of world-knowledge is absent. This signals student inference is limited to text-base representation; without the schema/background knowledge activation, their inference may be thin, lacking elaboration of the “why” and the “how” behind the narrative"
"\n IIR2: World Knowledge (Global) Only"
    "\n Students generate an inference connecting the question primarily with their background knowledge. Use of text-base information is implicit. This signals their 'script-based' reasoning—they answer based on how the world usually operates without explicitly integrating how the story specifically describes it."
"\n IIR3: Combination (Global and Local)"
    "\n Students generate an inference connecting the question with both world/background knowledge AND information explicitly mentioned in the text (text-based information). In other words, students go beyond merely identifying text-base information,  anchoring it within a broader real-world schema (i.e., background knowledge) to explain the why or how behind the narrative. This achieves the highest level of coherence (“the situation model”, Kintsch, 1988, 1998) because the student integrates the text-base information and their prior knowledge to generate a novel, unified interpretation."
"\n\n Note: Each simulated student will either evaluate a comic OR text-only scenario that will be given by the user prompt."

"\n Each simulated student also will specify one of the 4 'comic experience levels' that that they have, given by the user prompt."
"\n The comic experience levels are:"
"\n 'No experience', 'A little experience', 'More than a little / less than a lot', 'A lot' "
"\n The students will also give brief responses."
"\n The students will answer two pairs of questions: (1a)/(1b) and (2a)/(2b). Be sure that each pair maintains contintuity or relevance between the (a) and (b) parts, while also allowing some flexibility for the student to potentially give creative answers as well. Follow the waypoints"
)


In [4]:
# ============================================================
# Stories, comic paths & scenario data
# ============================================================
cat_text = """
There was a cat that lived on the streets that really wanted to find a home.
But nobody wanted him. “Stinky cat”, said one boy to the cat.  “Yucky cat”, said a girl to the cat.

Then on one cloudy day, the cat walked down a different street.
He saw a boy with red hair sitting by himself, crying.  The cat wanted to make him feel better.

So the cat jumped up on the bench, and walked onto his lap.
The cat looked straight up at the red headed boy and said “Meow”.

The boy looked down at the dirty cat. The boy pet the cat.
The cat said “purrrrr”.  Then some boys (KIDS)  started laughing at the red headed boy.
“Hahaha.  Now you’re just as yucky as that dirty cat” shouted one of the boys.

The boy hated being laughed at, so he quickly put the cat down and stopped petting the cat.
He shouted at the cat, “go away cat!  You’re making them laugh at me!”
“Hey, you can come back and play basketball with us again, but after you wash your hands” shouted a boy.

(THEN) The red headed boy, and his friends, walked back home together and came across the cat again.
As the cat was sitting in the rain, wet, cold, and shaking, the boys were being mean to it and laughed at the cat.
“Haha it’s about time you get a shower you stinky cat!” said one of the boys.
The red headed boy saw how sad the cat was.  He walked over to it.
“Don’t think I’ll let you play basketball with us again if you touch that dirty cat”, said one of the boys.
The red headed boy bent down, and pet the cat anyways.
He said to the cat, “hey there real friend.  My name is Bruno, and I’m going to take you to your new home, my home”.


"""

lying_text = """
Two girls, Sarah and Rachel, have been best friends since they were toddlers.
They are now in middle school, and are taking the same math class together.

•“Hey Sarah, have you studied for that math test yet?” asked Rachel.
“Yeah.  I’ve studied a lot for it.  What about you?” asked Sarah.

“I haven’t studied at all.  But I feel confident that I will do well anyways”, said Rachel.

It is the day of the test now.  They are sitting at their desks.
“Alright everyone, I am about to pass your tests out.  Remember, no cheating.
You have 40 minutes to finish”, says the teacher.

Sarah says to herself,  “Hmm I better make sure I read each question carefully, so that I don’t miss anything, and answer any questions wrong”.

Sarah looks over at her best friend Rachel.  She sees Rachel bubbling in every answer quickly on her answer sheet. Sarah says to herself, “But how could she know the answer to each question so quickly?  She didn’t even study.  She’s just probably guessing then.”

Then Sarah sees Rachel looking over the shoulder of the kid who is sitting right in front of her. It looks like she is copying his answer sheet.

Rachel notices that Sarah is looking at her.  Rachel whispers to Sarah, “Sarah shhhhh.  Don’t tell on me.  I’m your best friend”.

“Is something going on Sarah?” asks the teacher.  “Nothing is going on Ms. Craig,” answers Sarah.  Sarah continues taking the test without looking up again.


"""

stealing_text = """
Billy loved bikes.  He was riding his bike to the beach one day, when he came across the most beautiful bike he had ever seen.  He said to the boy that was riding the beautiful bike, “wow, that’s such a beautiful bike!”  “Thanks!” said the boy.

“Hey, how much did you pay for that bike?” asked Billy.  “It was $300”, answered the boy.

Billy rode his bike home.  He thought to himself, “hmmm, maybe my parents will buy me that bike for my birthday”.

Billy got home. He says to his parents,“hey mom and dad.  For my birthday I would like a new bike pretty please!”.  “No Billy.  You already have a nice bike and you don’t need a new expensive one”, said his father.

Billy felt sad, and mad.  So he took his bike, and rode it to an ice cream shop.

Billy walked out of the ice cream shop with ice cream, and found a table to sit outside. “Ice cream, you always make me feel better” Billy said to himself.  Next to him was another table, where a lady was sitting.

As Billy was eating his ice cream, the lady left and forgot her wallet.  Billy went and grabbed her wallet.  He opened her wallet.  “Wow, is this really $300? With that money, I could buy that red bike!”  Billy said.

The lady that was sitting at the table next to him came back, looking for her wallet.  She said, “hello there, have you seen my wallet?  I can’t find it anywhere.”  “No, I don’t know where it is”, said Billy.


"""

# Comic versions (PDFs in the project folder; Gemini reads them directly)
cat_comic_img = "Cat.pdf"
lying_comic_img = "Lying.pdf"
stealing_comic_img = "Stealing.pdf"

# Scenario lookup: lets us shuffle scenarios and question pairs independently
scenarios_data = {
    "Cat": {
        "text": cat_text,
        "comic": cat_comic_img,
        "pairs": [
            {
                "questions": "(1a).Why did the red headed boy keep the cat?\n(1b).What made you think of that answer?",
                "keys": ["cat_1", "cat_2"]
            },
            {
                "questions": "(2a).What lesson could someone learn from this story?\n(2b).What made you think of that answer?",
                "keys": ["cat_3", "cat_4"]
            }
        ]
    },
    "Lying": {
        "text": lying_text,
        "comic": lying_comic_img,
        "pairs": [
            {
                "questions": "(1a).Why doesn’t Sarah tell the teacher that Rachel is looking at another student’s answers?\n(1b).What made you think of that answer?",
                "keys": ["lying_1", "lying_2"]
            },
            {
                "questions": "(2a).What lesson could someone learn from this story?\n(2b).What made you think of that answer?",
                "keys": ["lying_3", "lying_4"]
            }
        ]
    },
    "Stealing": {
        "text": stealing_text,
        "comic": stealing_comic_img,
        "pairs": [
            {
                "questions": "(1a).Why did Billy keep the lady’s wallet?\n(1b).What made you think of that answer?",
                "keys": ["stealing_1", "stealing_2"]
            },
            {
                "questions": "(2a).What lesson could someone learn from this story?\n(2b).What made you think of that answer?",
                "keys": ["stealing_3", "stealing_4"]
            }
        ]
    }
}


In [5]:
# ============================================================
# Output schema, sampling weights & generation instructions
# ============================================================
# The 12 response keys, in CSV column order
ITEM_KEYS = [
    "cat_1", "cat_2", "cat_3", "cat_4",
    "lying_1", "lying_2", "lying_3", "lying_4",
    "stealing_1", "stealing_2", "stealing_3", "stealing_4",
]

# Uniform waypoint sampling (25% each)
level_proportions = {"IIR0": 0.25, "IIR1": 0.25, "IIR2": 0.25, "IIR3": 0.25}

instructions = """
==================== INSTRUCTIONS ====================
1. You MUST answer every single item. Do not leave any key empty.
2. Write exactly as a student at the specified IIR level would write.
3. Keep responses short and natural — like a real student, not an essay and only a couple of sentences.
4. Do not use perfect grammar or overly sophisticated vocabulary.
5. The meta-reasoning answers (2a, 2b) should reference BOTH the story
   AND personal experience depending on the IIR level.
6. Do not mention the IIR level anywhere.
7. Maintain continuity between part (a) and (b) for each pair of questions, that is appropriate for each waypoint level.
8. Have the student generated responses be as thorough as what is necessary to follow the waypoint guidelines.
9. Output valid JSON only — no extra text, no markdown, no code fences.
"""


In [6]:
# ============================================================
# Helper functions
# ============================================================

@lru_cache(maxsize=None)
def load_comic(path):
    """Read a comic PDF once and cache its bytes."""
    with open(path, "rb") as f:
        return f.read()


def build_profile_prompt(level):
    profile = profiles[level]
    return (
        "==================== TARGET STUDENT PROFILE ====================\n"
        f"IIR Developmental Level: {profile['label']}\n"
        "CONSTRUCT MAP DEFINITION:\n"
        f"{profile['definition']}\n"
    )


def generate_sample(total_n, proportions=level_proportions):
    """Build a list of `total_n` waypoint labels matching the target proportions."""
    counts = {level: round(prop * total_n) for level, prop in proportions.items()}

    # Fix rounding so the counts sum to exactly total_n
    diff = total_n - sum(counts.values())
    if diff != 0:
        for level in random.sample(list(counts), abs(diff)):
            counts[level] += 1 if diff > 0 else -1

    sample = []
    for level, count in counts.items():
        sample.extend([level] * count)
    return sample


def parse_response(json_str):
    """Parse the model's JSON into the 12 item keys; missing/invalid -> None."""
    result = {key: None for key in ITEM_KEYS}
    if json_str is None:
        return result
    try:
        parsed = json.loads(json_str)
    except Exception as e:
        print(f"JSON parse error: {e}")
        return result
    for key in ITEM_KEYS:
        val = parsed.get(key)
        if val is not None:
            result[key] = str(val)
    return result


In [7]:
# ============================================================
# Prompt builder: one student -> one multi-scenario prompt
# ============================================================

def build_prompt(level, format_type):
    # Randomize scenario order for this student
    scenario_names = ["Cat", "Lying", "Stealing"]
    random.shuffle(scenario_names)

    scenarios_prompt_text = ""
    json_keys_ordered = []
    document_parts = []

    for i, s_name in enumerate(scenario_names):
        s_data = scenarios_data[s_name]

        # Randomize the order of the two question pairs
        pairs = s_data["pairs"].copy()
        random.shuffle(pairs)
        shuffled_questions = f"{pairs[0]['questions']}\n{pairs[1]['questions']}"
        json_keys_ordered.extend(pairs[0]["keys"])
        json_keys_ordered.extend(pairs[1]["keys"])

        scenarios_prompt_text += f"==================== SCENARIO {i+1}: {s_name.upper()} ====================\n"
        if format_type == "text":
            scenarios_prompt_text += f"STORY:\n{s_data['text']}\n\n"
        elif format_type == "comic":
            document_parts.append(
                Part.from_data(data=load_comic(s_data["comic"]), mime_type="application/pdf")
            )
        scenarios_prompt_text += f"QUESTIONS:\n{shuffled_questions}\n\n"

    # Dynamic JSON schema, in the (shuffled) key order
    json_format_str = "{\n" + ",\n".join(f'    "{key}": "..."' for key in json_keys_ordered) + "\n}"
    response_format = f"""
==================== RESPONSE FORMAT ====================
Return a single valid JSON object with exactly these 12 keys in the exact order shown below.
Each value should be a short open-ended text response as the student would write it.
Do not include any explanation or text outside the JSON.

{json_format_str}
"""

    if format_type == "comic":
        intro = (
            "Below are 3 comic stories attached as documents in the following order: "
            f"{scenario_names[0]}, {scenario_names[1]}, and {scenario_names[2]}. "
            "Please read the comics and answer the corresponding questions for each."
        )
    else:
        intro = "Below are 3 different scenarios. Please read the story and answer the questions for each."

    prompt_text = (
        build_profile_prompt(level) + "\n\n" +
        intro + "\n\n" +
        scenarios_prompt_text +
        response_format + "\n\n" +
        instructions
    )

    prompt_user = document_parts + [prompt_text] if format_type == "comic" else prompt_text
    return System_Prompt, prompt_user


In [8]:
# ============================================================
# API caller (with retry / exponential backoff)
# ============================================================

def call_api(prompt_system, prompt_user, max_retries=3):
    model = GenerativeModel(model_name=MODEL_NAME, system_instruction=prompt_system)
    for attempt in range(max_retries):
        try:
            response = model.generate_content(
                prompt_user,
                generation_config={
                    "temperature": 1.0,
                    "response_mime_type": "application/json",
                },
            )
            return response.text
        except Exception as e:
            print(f"  [Attempt {attempt+1}] Error: {e}")
            time.sleep(2 ** attempt)
    print(f"API call failed after {max_retries} retries")
    return None


In [9]:
# ============================================================
# Main pipeline
# ============================================================

def _save_progress(payload):
    """Atomically write the progress file (temp + replace) to avoid corruption."""
    tmp = PROGRESS_FILE + ".tmp"
    with open(tmp, "w") as f:
        json.dump(payload, f)
    os.replace(tmp, PROGRESS_FILE)


def run_pipeline(total_n=100, seed=42, resume=True):
    random.seed(seed)

    # Assign a waypoint to each student, then split each waypoint ~50/50 comic/text
    level_assignments = generate_sample(total_n)

    level_counts = {}
    for lvl in level_assignments:
        level_counts[lvl] = level_counts.get(lvl, 0) + 1

    format_pools = {}
    for lvl, count in level_counts.items():
        num_comic = count // 2
        pool = ["comic"] * num_comic + ["text"] * (count - num_comic)
        random.shuffle(pool)
        format_pools[lvl] = pool
    format_assignments = [format_pools[lvl].pop() for lvl in level_assignments]

    raw_json = [None] * total_n
    raw_answers = [None] * total_n
    start_i = 0

    # Resume from a prior run if the parameters match
    if resume and os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            prog = json.load(f)
        if (prog.get("total_n") == total_n and prog.get("seed") == seed
                and prog.get("exact_ratio") is True):
            level_assignments = prog["level_assignments"]
            format_assignments = prog["format_assignments"]
            raw_json = prog["raw_json"]
            raw_answers = prog["raw_answers"]
            start_i = prog["completed"]
            print(f"==> Resuming from progress file: {start_i}/{total_n} already completed.")
        else:
            print("==> Progress file found but parameters differ. Starting fresh.")

    resp_ids = [f"R{i+1:04d}" for i in range(total_n)]

    print(f"Level distribution: {pd.Series(level_assignments).value_counts().to_dict()}")
    df_verify = pd.DataFrame({"Level": level_assignments, "Format": format_assignments})
    print("Format breakdown per level:")
    print(df_verify.groupby(["Level", "Format"]).size().unstack(fill_value=0).to_dict("index"))

    if start_i >= total_n:
        print("All responses already generated. Skipping API calls.")
    else:
        for i in range(start_i, total_n):
            level, fmt = level_assignments[i], format_assignments[i]
            print(f"\r[{i+1}/{total_n}] Level: {level} | Format: {fmt}", end="")

            prompt_system, prompt_user = build_prompt(level, fmt)
            raw_json[i] = call_api(prompt_system, prompt_user)
            raw_answers[i] = parse_response(raw_json[i])

            _save_progress({
                "total_n": total_n,
                "seed": seed,
                "exact_ratio": True,
                "level_assignments": level_assignments,
                "format_assignments": format_assignments,
                "raw_json": raw_json,
                "raw_answers": raw_answers,
                "completed": i + 1,
            })
            time.sleep(0.5)
        print("\nAPI calls complete.")

    # Assemble the output table
    answer_df = pd.DataFrame({
        "respondent_id": resp_ids,
        "profile_level": level_assignments,
        "format": format_assignments,
    })
    for key in ITEM_KEYS:
        answer_df[key] = [ans.get(key) if ans else None for ans in raw_answers]

    answer_df.to_csv(os.path.join(OUTPUT_DIR, OUTPUT_CSV), index=False)
    with open(os.path.join(OUTPUT_DIR, "IIR_raw_json.json"), "w") as f:
        json.dump(raw_json, f)

    if os.path.exists(PROGRESS_FILE):
        os.remove(PROGRESS_FILE)

    print(f"Saved to: {OUTPUT_DIR}")
    print(f"  - {OUTPUT_CSV}  (all responses)")
    print("  - IIR_raw_json.json    (raw API responses)")
    return answer_df


In [10]:
# ============================================================
# Run the pipeline
# ============================================================
# Change total_n to your desired sample size (multiples of 8 keep the
# per-waypoint comic/text split even).
df_final = run_pipeline(total_n=16, seed=53, resume=True)

# Preview the generated table
pd.set_option("display.max_columns", None)
df_final.head()


Level distribution: {'IIR0': 4, 'IIR1': 4, 'IIR2': 4, 'IIR3': 4}
Format breakdown per level:
{'IIR0': {'comic': 2, 'text': 2}, 'IIR1': {'comic': 2, 'text': 2}, 'IIR2': {'comic': 2, 'text': 2}, 'IIR3': {'comic': 2, 'text': 2}}
[1/16] Level: IIR0 | Format: comic

c:\Users\david\Documents\IIR\.venv\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


[2/16] Level: IIR0 | Format: text

c:\Users\david\Documents\IIR\.venv\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


[16/16] Level: IIR3 | Format: comic
API calls complete.
Saved to: IIR_outputs
  - IIR_raw_answers_Master4.csv  (all responses)
  - IIR_raw_json.json    (raw API responses)


,respondent_id,profile_level,format,cat_1,cat_2,cat_3,cat_4,lying_1,lying_2,lying_3,lying_4,stealing_1,stealing_2,stealing_3,stealing_4
0,R0001,IIR0,comic,I dont know.,I just don't know the answer to it.,always eat your vegetables before dessert.,thats what my dad tells me every night.,Because her shirt is pink.,I like pink shirts and she is wearing one.,idk.,nothing.,because wallets hold pictures of peoples dogs.,my mom has a picture of our dog in her wallet.,I don't really know.,Cause I just dont know what a lesson is.
1,R0002,IIR0,text,Because red hair is cool.,My cousin has red hair.,I don't know.,It's just a story about a cat.,I have no idea.,Because I wasn't there.,You should use a pencil on math tests.,Cause you have to bubble in the answers on the...,"I don't know, maybe he just likes wallets.",I just guessed.,Ice cream is really good.,Because he ate ice cream at the shop.
2,R0003,IIR0,text,I don't know.,I just guessed.,Cats are stinky.,My dog is stinky sometimes too.,She likes to sit at desks.,Because they are in middle school.,Math tests are 40 minutes.,Because the teacher said it.,I don't know.,I just don't know.,Bikes cost 300 dollars.,It said 300 in the story.
3,R0004,IIR0,comic,Because cats like to play with yarn.,My cat at home plays with yarn a lot.,I don't know what lesson it is.,I just don't know.,I don't know.,I didn't know what to put.,Don't go to school.,Because school is boring sometimes.,He likes to eat ice cream.,Ice cream is really good.,I don't know.,Because I couldn't think of anything.
4,R0005,IIR1,text,He kept the cat because he saw how sad the cat...,The text says 'The red headed boy saw how sad ...,You can take a dirty cat to your house if the ...,Because the story says the boys were laughing ...,Because Rachel whispered to her to not tell on...,I know because the story says Rachel said 'Don...,Don't tell the teacher when your friend is cop...,Because Sarah just said 'Nothing is going on M...,Because Billy wanted to use the $300 to buy th...,Because in the story Billy says 'With that mon...,Don't forget your wallet on the table when you...,Because the lady left her wallet at the table ...
